In [1]:
import nltk, math, random
import numpy as np
from collections import defaultdict, Counter
from tqdm import tqdm

# Download data if not present
nltk.download('brown', quiet=True)
nltk.download('universal_tagset', quiet=True)

True

In [2]:
USE_BROWN_TAGSET = False

from nltk.corpus import brown

if USE_BROWN_TAGSET:
    tagged_sents = brown.tagged_sents(tagset=None)
else:
    tagged_sents = brown.tagged_sents(tagset='universal')

print(f"Using {'Brown (full)' if USE_BROWN_TAGSET else 'Universal'} tagset")
print(f"Total sentences: {len(tagged_sents):,}")

Using Universal tagset
Total sentences: 57,340


## 1. Data exploration & statistics

In [3]:
sample_sents = tagged_sents[:3]
for i, sent in enumerate(sample_sents, 1):
    print(f"Sentence {i}: {sent}\n")

total_words = sum(len(s) for s in tagged_sents)
vocab_all = set(word.lower() for sent in tagged_sents for word, tag in sent)
tags_all = set(tag for sent in tagged_sents for word, tag in sent)
print(f"Total sentences: {len(tagged_sents):,}")
print(f"Total words (tokens): {total_words:,}")
print(f"Vocabulary size (lowercased): {len(vocab_all):,}")
print(f"Unique tags in corpus: {len(tags_all)}")

Sentence 1: [('The', 'DET'), ('Fulton', 'NOUN'), ('County', 'NOUN'), ('Grand', 'ADJ'), ('Jury', 'NOUN'), ('said', 'VERB'), ('Friday', 'NOUN'), ('an', 'DET'), ('investigation', 'NOUN'), ('of', 'ADP'), ("Atlanta's", 'NOUN'), ('recent', 'ADJ'), ('primary', 'NOUN'), ('election', 'NOUN'), ('produced', 'VERB'), ('``', '.'), ('no', 'DET'), ('evidence', 'NOUN'), ("''", '.'), ('that', 'ADP'), ('any', 'DET'), ('irregularities', 'NOUN'), ('took', 'VERB'), ('place', 'NOUN'), ('.', '.')]

Sentence 2: [('The', 'DET'), ('jury', 'NOUN'), ('further', 'ADV'), ('said', 'VERB'), ('in', 'ADP'), ('term-end', 'NOUN'), ('presentments', 'NOUN'), ('that', 'ADP'), ('the', 'DET'), ('City', 'NOUN'), ('Executive', 'ADJ'), ('Committee', 'NOUN'), (',', '.'), ('which', 'DET'), ('had', 'VERB'), ('over-all', 'ADJ'), ('charge', 'NOUN'), ('of', 'ADP'), ('the', 'DET'), ('election', 'NOUN'), (',', '.'), ('``', '.'), ('deserves', 'VERB'), ('the', 'DET'), ('praise', 'NOUN'), ('and', 'CONJ'), ('thanks', 'NOUN'), ('of', 'ADP'),

## 2. Split data (train/dev/test)

In [5]:
random.seed(42)
sents = list(tagged_sents)
random.shuffle(sents)

N = len(sents)
train_end = int(0.8 * N)
dev_end = int(0.9 * N)

train_sents = sents[:train_end]
dev_sents = sents[train_end:dev_end]
test_sents = sents[dev_end:]

print(f"Train: {len(train_sents):,}  Dev: {len(dev_sents):,}  Test: {len(test_sents):,}")

Train: 45,872  Dev: 5,734  Test: 5,734


## 3. Preprocessing — vocabulary, mappings, <UNK> handling

In [6]:
# Build vocab and tag sets from training data
MIN_FREQ = 1  # words with frequency <= MIN_FREQ will be mapped to <UNK> (you can raise this)

word_counts = Counter()
tag_set = set()

for sent in train_sents:
    for w, t in sent:
        word_counts[w.lower()] += 1
        tag_set.add(t)

# Construct vocab: words with freq > MIN_FREQ
vocab = {w for w, c in word_counts.items() if c > MIN_FREQ}
vocab.add('<UNK>')

# Tag set: include START and END special tags
tags = set(tag_set)
tags.add('<START>')
tags.add('<END>')

word2idx = {w: i for i, w in enumerate(sorted(vocab))}
tag2idx = {t: i for i, t in enumerate(sorted(tags))}
idx2tag = {i: t for t, i in tag2idx.items()}

print(f"Vocabulary size (after cutoff): {len(vocab)}") 
print(f"Number of tags (including START/END): {len(tags)}")

Vocabulary size (after cutoff): 24755
Number of tags (including START/END): 14


## 4. HMM implementation (log probs, add-k smoothing, Viterbi)

In [7]:
class HMM:
    def __init__(self, tags, vocab, k=1e-5):
        self.k = k  # smoothing
        self.tags = set(tags)
        self.vocab = set(vocab)
        
        # counts
        self.transition = defaultdict(lambda: defaultdict(int))
        self.emission = defaultdict(lambda: defaultdict(int))
        self.tag_counts = defaultdict(int)
        self.initial_counts = defaultdict(int)
        
        # log prob tables (filled after training)
        self.log_trans = {}
        self.log_emit = {}
        self.log_initial = {}
        
        # Most frequent tag overall (for backoff)
        self.most_freq_tag = None
    
    def train(self, tagged_sentences):
        # counting
        for sent in tqdm(tagged_sentences, desc='Counting'):
            if not sent: 
                continue
            prev = '<START>'
            # count initial tag
            self.initial_counts[sent[0][1]] += 1
            for w, t in sent:
                w = w.lower()
                self.tag_counts[t] += 1
                self.emission[t][w] += 1
                self.transition[prev][t] += 1
                prev = t
            # end transition
            self.transition[prev]['<END>'] += 1
        
        # record most frequent tag for fallback
        if self.tag_counts:
            self.most_freq_tag = max(self.tag_counts.items(), key=lambda x: x[1])[0]
        
        # compute log-probabilities with smoothing
        self._compute_logs()
    
    def _compute_logs(self):
        all_tags = list(self.tags)
        
        # initial log probs
        total_init = sum(self.initial_counts.values())
        for t in all_tags:
            if t in ['<START>', '<END>']: 
                continue
            cnt = self.initial_counts.get(t, 0)
            prob = (cnt + self.k) / (total_init + self.k * len(all_tags))
            self.log_initial[t] = math.log(prob)
        
        # transition log probs
        for prev in set(list(self.transition.keys()) + all_tags):
            self.log_trans[prev] = {}
            total = sum(self.transition[prev].values())
            for t in all_tags:
                cnt = self.transition[prev].get(t, 0)
                prob = (cnt + self.k) / (total + self.k * len(all_tags))
                self.log_trans[prev][t] = math.log(prob)
        
        # emission log probs
        V = len(self.vocab)
        for t in all_tags:
            self.log_emit[t] = {}
            total = self.tag_counts.get(t, 0)
            for w in self.vocab:
                cnt = self.emission[t].get(w, 0)
                prob = (cnt + self.k) / (total + self.k * V)
                self.log_emit[t][w] = math.log(prob)
    
    def get_emission_log(self, tag, word):
        w = word.lower()
        if w not in self.vocab:
            # fallback: if suffix heuristic helps, try suffix mapping (e.g., -ing => VERB)
            # but here use <UNK> emission if present; else backoff to most frequent tag probability
            w = '<UNK>'
        return self.log_emit.get(tag, {}).get(w, math.log(self.k))
    
    def get_transition_log(self, prev, curr):
        return self.log_trans.get(prev, {}).get(curr, math.log(self.k))
    
    def viterbi(self, words):
        n = len(words)
        tags_list = [t for t in self.tags if t not in ['<START>', '<END>']]
        # initialize
        V = [defaultdict(lambda: float('-inf')) for _ in range(n)]
        backpointer = [dict() for _ in range(n)]
        
        # first word
        for t in tags_list:
            V[0][t] = self.log_initial.get(t, math.log(self.k)) + self.get_emission_log(t, words[0])
            backpointer[0][t] = '<START>'
        
        # recursion
        for i in range(1, n):
            for t in tags_list:
                best_score = float('-inf')
                best_prev = None
                emit = self.get_emission_log(t, words[i])
                for p in tags_list:
                    score = V[i-1][p] + self.get_transition_log(p, t) + emit
                    if score > best_score:
                        best_score = score
                        best_prev = p
                V[i][t] = best_score
                backpointer[i][t] = best_prev
        
        # termination: add transition to <END>
        best_last, best_score = None, float('-inf')
        for t in tags_list:
            score = V[n-1][t] + self.get_transition_log(t, '<END>')
            if score > best_score:
                best_score = score
                best_last = t
        
        # backtrack
        path = [best_last]
        for i in range(n-1, 0, -1):
            prev = backpointer[i][path[0]]
            path.insert(0, prev)
        return path

print('HMM class defined')

HMM class defined


## 5. Train HMM on training data

In [8]:
# Build and train the HMM
hmm = HMM(tags=tags, vocab=vocab, k=1e-5)
hmm.train(train_sents)

print('Training done.')

Counting: 100%|██████████| 45872/45872 [00:00<00:00, 72823.41it/s]


Training done.


## 6. Inspect probabilities (examples)

In [9]:
# Example initial probabilities (top 10)
init_list = sorted([(t, math.exp(p)) for t, p in hmm.log_initial.items()], key=lambda x: x[1], reverse=True)
for t, p in init_list[:10]:
    print(f"P({t}) = {p:.6e}")

# Example transition and emission
print('\nExample transitions:')
for prev in ['<START>'] + list(sorted(tag for tag in tags)[:5]):
    row = [(curr, math.exp(hmm.get_transition_log(prev, curr))) for curr in list(sorted(tag for tag in tags)[:5])]
    print(prev, row)

P(DET) = 2.136597e-01
P(PRON) = 1.595963e-01
P(NOUN) = 1.415024e-01
P(ADP) = 1.237138e-01
P(ADV) = 9.042553e-02
P(.) = 8.702476e-02
P(CONJ) = 4.928933e-02
P(VERB) = 4.582316e-02
P(PRT) = 3.712504e-02
P(ADJ) = 3.483607e-02

Example transitions:
<START> [('.', 0.08702476451466108), ('<END>', 2.1799790655476725e-10), ('<START>', 2.1799790655476725e-10), ('ADJ', 0.03483606568544975), ('ADP', 0.12371381218782847)]
. [('.', 0.10650139559919064), ('<END>', 0.38109055084669435), ('<START>', 8.484138892244818e-11), ('ADJ', 0.028905461290719504), ('ADP', 0.06540422680515673)]
<END> [('.', 0.07142857142857141), ('<END>', 0.07142857142857141), ('<START>', 0.07142857142857141), ('ADJ', 0.07142857142857141), ('ADP', 0.07142857142857141)]
<START> [('.', 0.08702476451466108), ('<END>', 2.1799790655476725e-10), ('<START>', 2.1799790655476725e-10), ('ADJ', 0.03483606568544975), ('ADP', 0.12371381218782847)]
ADJ [('.', 0.10018594266975037), ('<END>', 0.0003718856072582524), ('<START>', 1.4875418340162749

## 7. Test on example sentences (quick)

In [10]:
examples = [
    ['The', 'dog', 'runs', 'quickly'],
    ['A', 'beautiful', 'cat', 'sleeps'],
    ['They', 'are', 'running', 'fast']
]

for ex in examples:
    tags_pred = hmm.viterbi(ex)
    print('\nSentence:', ' '.join(ex))
    print('Predicted:', tags_pred)


Sentence: The dog runs quickly
Predicted: ['DET', 'NOUN', 'VERB', 'ADV']

Sentence: A beautiful cat sleeps
Predicted: ['DET', 'ADJ', 'NOUN', '.']

Sentence: They are running fast
Predicted: ['PRON', 'VERB', 'VERB', 'ADV']


## 8. Evaluation (dev & test sets)

In [11]:
def evaluate(hmm, dataset, max_sents=None):
    correct = 0
    total = 0
    for i, sent in enumerate(dataset):
        if max_sents and i >= max_sents:
            break
        words = [w for w, t in sent]
        true_tags = [t for w, t in sent]
        pred_tags = hmm.viterbi(words)
        for a, b in zip(true_tags, pred_tags):
            if a == b:
                correct += 1
            total += 1
    return correct / total * 100 if total else 0.0

print('Evaluating on development subset (500 sentences)...')
dev_acc = evaluate(hmm, dev_sents, max_sents=500)
print(f'Dev accuracy (subset): {dev_acc:.2f}%')

print('Evaluating on test subset (500 sentences)...')
test_acc = evaluate(hmm, test_sents, max_sents=500)
print(f'Test accuracy (subset): {test_acc:.2f}%')

Evaluating on development subset (500 sentences)...
Dev accuracy (subset): 94.58%
Evaluating on test subset (500 sentences)...
Test accuracy (subset): 94.65%


## 9. Error analysis: confusion matrix (top confusions)

In [12]:
from collections import defaultdict
conf = defaultdict(lambda: defaultdict(int))
for sent in dev_sents[:500]:
    words = [w for w,t in sent]
    true = [t for w,t in sent]
    pred = hmm.viterbi(words)
    for a,b in zip(true,pred):
        conf[a][b] += 1

confusions = []
for a in conf:
    for b in conf[a]:
        if a != b and conf[a][b] > 0:
            confusions.append((a,b,conf[a][b]))

confusions.sort(key=lambda x: x[2], reverse=True)
print('Top confusions:')
for a,b,c in confusions[:20]:
    print(f'{a} -> {b}: {c}')

Top confusions:
VERB -> NOUN: 53
NOUN -> X: 45
NOUN -> ADJ: 41
NOUN -> NUM: 37
NOUN -> VERB: 35
PRT -> ADP: 23
ADP -> ADV: 22
ADJ -> ADV: 20
NOUN -> PRON: 19
VERB -> ADP: 19
ADV -> ADJ: 16
ADJ -> NOUN: 15
ADP -> PRT: 14
NOUN -> .: 14
NOUN -> DET: 13
NOUN -> ADV: 13
NOUN -> CONJ: 11
NOUN -> ADP: 11
VERB -> ADV: 11
ADJ -> DET: 11


## 10. Unknown words analysis

In [13]:
unknowns = []
for sent in test_sents:
    for w,t in sent:
        if w.lower() not in hmm.vocab:
            unknowns.append((w,t))

print(f'Unknown words in test set: {len(unknowns):,} (examples)')
for u in unknowns[:30]:
    print(u)

Unknown words in test set: 4,119 (examples)
('understandingly', 'ADV')
('Stratton', 'NOUN')
('distastefully', 'ADV')
("supervisors'", 'NOUN')
('Catskills', 'NOUN')
('mid-October', 'NOUN')
('23A', 'NUM')
('Tobin', 'NOUN')
('Barrington', 'NOUN')
('persisting', 'VERB')
('schoolers', 'NOUN')
('blatancy', 'NOUN')
('Pasley', 'NOUN')
('ostentatious', 'ADJ')
('Belshazzar', 'NOUN')
('feasts', 'NOUN')
('fraternized', 'VERB')
('jowl', 'NOUN')
('by-passed', 'VERB')
('Pincian', 'ADJ')
('shockwave', 'NOUN')
('Galilee', 'NOUN')
('isocyanate', 'NOUN')
('anhydrously', 'ADV')
('foaming', 'NOUN')
('adapters', 'NOUN')
('reformatory', 'NOUN')
('letterman', 'NOUN')
('fictive', 'ADJ')
('cross-fertilization', 'NOUN')
